# 16 — Chain-of-Thought RAG

> **Tier 3 | Query Handling**

## The Problem

Complex questions require **sequential reasoning** — each intermediate conclusion
should inform what to look for next:

```
Q: "How do feedback loops in the Arctic amplify global warming?"

You can't answer this in one shot. You need to reason step by step:
  Step 1: What is the Arctic's current state and baseline ice coverage?
  Step 2: What feedback mechanisms operate in the Arctic? (albedo, permafrost, etc.)
  Step 3: How do those feedbacks couple back to global temperature?
  Step 4: What does evidence show about acceleration?
  → Synthesise across all four findings
```

Simple RAG and even Query Decomposition retrieve **in parallel** — they can't
use what they find in step 1 to sharpen the query for step 2.

## Chain-of-Thought RAG Solution

1. **Plan** — LLM decomposes the question into an ordered chain of reasoning steps
2. **Loop** — for each step:
   - Formulate a retrieval query from the step description **plus prior findings**
   - Retrieve top-K chunks
   - Reason over retrieved context → produce an intermediate *finding*
3. **Synthesise** — LLM combines all intermediate findings into the final answer

The key is that each retrieval query is **dynamically enriched** by the accumulated
context from all previous steps.

## Flow Diagram

```mermaid
flowchart TD
    Q(["❓ Complex Question"])
    Q --> PLAN["🧠 LLM Planner\n→ Step 1 … Step N (ordered chain)"]

    subgraph LOOP ["🔁  Sequential Retrieve-and-Reason Loop"]
        direction TB
        S1["Step 1\nQuery = step_desc\n→ retrieve → reason → finding₁"]
        S2["Step 2\nQuery = step_desc + finding₁\n→ retrieve → reason → finding₂"]
        SN["Step N\nQuery = step_desc + all prior findings\n→ retrieve → reason → findingₙ"]
        S1 --> S2 --> SN
    end

    PLAN --> LOOP

    LOOP --> SYNTH["🟠 Strands Agent\nSynthesise all findings → final answer"]
    SYNTH --> ANS(["✅ Progressively-grounded answer"])

    style LOOP  fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style SYNTH fill:#ffedd5,stroke:#f97316,color:#7c2d12
```

## Step 1 — Install Dependencies

In [1]:
import subprocess, sys
packages = ["boto3","qdrant-client","opensearch-py","requests-aws4auth",
            "strands-agents","pypdf","python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")

All packages ready.


## Step 2 — Imports & Configuration

In [2]:
import os, json, time, uuid, re
from typing import List, Dict, Optional

import boto3, pypdf
from dotenv import load_dotenv
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)

AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME = "chain_of_thought_rag_16"
EMBEDDING_DIM   = 1024
TOP_K           = 4    # chunks retrieved per reasoning step
MAX_STEPS       = 5    # cap on chain length
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
PDF_PATH        = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection : {COLLECTION_NAME}")
print(f"PDF exists : {os.path.exists(PDF_PATH)}")
print(f"AWS region : {AWS_REGION}")
print(f"Key ID     : {os.getenv('AWS_ACCESS_KEY_ID','NOT SET')[:12]}...")

Collection : chain_of_thought_rag_16
PDF exists : True
AWS region : us-east-1
Key ID     : ASIA4KPWEP6M...


## Step 3 — Vector Store

In [3]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists — skipping recreate')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5,
               query_filter=None) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k,
                query_filter=query_filter, with_payload=True)
            return [{'text': p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]
        return []

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0
        return 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        print(f'Deleted "{self.name}"')

print("VectorStore defined.")

VectorStore defined.


## Step 4 — Bedrock Helpers

In [4]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str) -> str:
    return str(Agent(model=_model, system_prompt="You are a helpful assistant.")(prompt))

test_emb = embed_text("chain of thought RAG test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Bedrock LLM ready.")

Embedding OK — dim=1024
Bedrock LLM ready.


## Step 5 — Connect & Index climate.pdf

In [5]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)

vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL,
    region=AWS_REGION
)
print(f"Backend: {vs._backend}")
vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

reader    = pypdf.PdfReader(PDF_PATH)
full_text = ''
page_map  = []
for pg_idx, page in enumerate(reader.pages):
    pg_text = (page.extract_text() or '') + '\n\n'
    page_map.extend([pg_idx + 1] * len(pg_text))
    full_text += pg_text

chunks = recursive_split(full_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"PDF pages: {len(reader.pages)}  |  Chunks: {len(chunks)}")

embs = embed_batch(chunks, label='[index]')
docs = []
for i, (chunk, emb) in enumerate(zip(chunks, embs)):
    char_offset = full_text.find(chunk[:40])
    page_num = page_map[min(char_offset, len(page_map)-1)] if char_offset >= 0 else 1
    docs.append({
        'text'     : chunk,
        'embedding': emb,
        'metadata' : {'chunk_idx': i, 'page_num': page_num, 'char_count': len(chunk)}
    })
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors into '{COLLECTION_NAME}'")

Qdrant Cloud: https://2210ff1c-7c49-4fec-91f4-01662586299c.us-east-1-1.aws.cloud.qdrant.io
Backend: qdrant_cloud
Created "chain_of_thought_rag_16" (dim=1024)
PDF pages: 13  |  Chunks: 66
  [index] 20/66
  [index] 40/66
  [index] 60/66
Indexed 66 vectors into 'chain_of_thought_rag_16'


## Step 6 — Chain-of-Thought Planner

Given the question, the LLM produces an **ordered** list of reasoning steps.
Each step is self-contained but builds on what came before.  
Steps are returned as a JSON array so they can be iterated programmatically.

In [6]:
def plan_cot_steps(question: str, max_steps: int = MAX_STEPS) -> List[str]:
    prompt = (
        f"Break the following question into {max_steps} or fewer ordered reasoning steps.\n"
        f"Each step should:\n"
        f"  - Be a focused sub-question or retrieval goal\n"
        f"  - Build logically on the previous step\n"
        f"  - Be answerable from a document corpus\n"
        f"Return ONLY a JSON array of strings — no explanation, no markdown fences.\n\n"
        f"Question: {question}\n\nJSON array:"
    )
    raw = llm(prompt).strip()
    raw = re.sub(r'^```[a-z]*\n?', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'```$', '', raw, flags=re.MULTILINE)
    try:
        steps = json.loads(raw.strip())
        if isinstance(steps, list):
            return [s for s in steps if isinstance(s, str) and s.strip()][:max_steps]
    except json.JSONDecodeError:
        # Fallback: extract quoted strings
        steps = re.findall(r'"([^"]+)"', raw)
        if steps:
            return steps[:max_steps]
    return [question]

# Demo
demo_q = "How do feedback loops in the Arctic amplify global warming?"
steps  = plan_cot_steps(demo_q)
print(f"Question: {demo_q}")
print(f"Chain ({len(steps)} steps):")
for i, s in enumerate(steps, 1):
    print(f"  {i}. {s}")

["What is Arctic amplification and why does the Arctic warm faster than other regions?", "How does sea ice loss create a positive feedback loop through the ice-albedo effect?", "How does permafrost thawing release greenhouse gases that further accelerate warming?", "How do changes in Arctic snow cover and water vapor contribute to additional warming feedbacks?", "How do these combined Arctic feedback loops interact to amplify global warming beyond the Arctic region?"]Question: How do feedback loops in the Arctic amplify global warming?
Chain (5 steps):
  1. What is Arctic amplification and why does the Arctic warm faster than other regions?
  2. How does sea ice loss create a positive feedback loop through the ice-albedo effect?
  3. How does permafrost thawing release greenhouse gases that further accelerate warming?
  4. How do changes in Arctic snow cover and water vapor contribute to additional warming feedbacks?
  5. How do these combined Arctic feedback loops interact to amplify 

## Step 7 — Sequential Retrieve-and-Reason Loop

For each step we:
1. **Formulate query**: `step_description + summary of prior findings` (so later steps are enriched by earlier ones)
2. **Retrieve** top-K chunks
3. **Reason**: LLM answers only that step, given retrieved context + prior findings → produces an *intermediate finding*

The intermediate finding is short (2–3 sentences) so it stays compact as it accumulates.

In [7]:
def cot_step(step_desc: str,
             prior_findings: List[str],
             top_k: int = TOP_K) -> Dict:
    # Build retrieval query: step description enriched with prior context
    if prior_findings:
        prior_summary = ' '.join(prior_findings[-2:])  # last 2 findings to stay focused
        retrieval_query = f"{step_desc}. Context so far: {prior_summary[:300]}"
    else:
        retrieval_query = step_desc

    # Retrieve
    qvec = embed_text(retrieval_query)
    hits = vs.search(qvec, top_k=top_k)

    # Build context
    context = '\n\n'.join(
        f"[p.{h['metadata'].get('page_num','?')}]\n{h['text']}" for h in hits
    )

    # Reason
    prior_block = ''
    if prior_findings:
        prior_block = ('Prior reasoning:\n' +
                       '\n'.join(f"  • {f}" for f in prior_findings) + '\n\n')

    reason_prompt = (
        f"{prior_block}"
        f"Context:\n{context}\n\n"
        f"Current reasoning step: {step_desc}\n\n"
        f"Answer ONLY this step in 2-3 sentences. "
        f"Cite page numbers (e.g. [p.3]). "
        f"If the context does not cover this step, say so briefly.\n\nFinding:"
    )
    finding = llm(reason_prompt).strip()

    return {
        'step'           : step_desc,
        'retrieval_query': retrieval_query,
        'hits'           : hits,
        'finding'        : finding,
        'pages'          : sorted({h['metadata'].get('page_num','?') for h in hits}),
    }

# Quick smoke test
result = cot_step(
    "What is the current state of Arctic sea ice?",
    prior_findings=[]
)
print(f"Step    : {result['step']}")
print(f"Pages   : {result['pages']}")
print(f"Finding : {result['finding']}")

The provided context does not contain any information about the current state of Arctic sea ice. The passages focus on satellite technology, weather forecasting methods, and meteorological data collection systems, with no mention of Arctic sea ice conditions.Step    : What is the current state of Arctic sea ice?
Pages   : [4, 6, 9, 12]
Finding : The provided context does not contain any information about the current state of Arctic sea ice. The passages focus on satellite technology, weather forecasting methods, and meteorological data collection systems, with no mention of Arctic sea ice conditions.


## Step 8 — Full CoT RAG Pipeline

In [8]:
def rag_cot(question: str,
            top_k: int = TOP_K,
            max_steps: int = MAX_STEPS,
            verbose: bool = True) -> Dict:
    t0 = time.time()

    # 1. Plan the reasoning chain
    steps = plan_cot_steps(question, max_steps)

    # 2. Sequential retrieve-and-reason
    step_results  = []
    prior_findings = []
    all_pages      = set()

    for i, step_desc in enumerate(steps):
        if verbose:
            print(f"  Step {i+1}/{len(steps)}: {step_desc[:70]}..." if len(step_desc) > 70
                  else f"  Step {i+1}/{len(steps)}: {step_desc}")
        result = cot_step(step_desc, prior_findings, top_k=top_k)
        step_results.append(result)
        prior_findings.append(result['finding'])
        all_pages.update(result['pages'])

    # 3. Synthesise across all findings
    chain_summary = '\n'.join(
        f"Step {i+1} — {r['step']}:\n{r['finding']}"
        for i, r in enumerate(step_results)
    )
    synth_prompt = (
        f"You have worked through the following reasoning chain to answer a question.\n"
        f"Each step retrieved relevant evidence and produced an intermediate finding.\n\n"
        f"{chain_summary}\n\n"
        f"Now write a complete, cohesive answer to the original question.\n"
        f"Cite page numbers (e.g. [p.3]) where relevant.\n\n"
        f"Original question: {question}\n\nFinal answer:"
    )
    final_answer = llm(synth_prompt)
    latency = (time.time() - t0) * 1000

    if verbose:
        print(f"\n{'='*65}")
        print(f"Q: {question}")
        print(f"Chain: {len(steps)} steps  |  Pages covered: {sorted(all_pages)}  |  Latency: {latency:.0f}ms")
        print(f"\nReasoning chain:")
        for i, r in enumerate(step_results):
            print(f"  [{i+1}] {r['step']}")
            print(f"       Pages: {r['pages']}")
            print(f"       Finding: {r['finding'][:120]}..." if len(r['finding']) > 120
                  else f"       Finding: {r['finding']}")
        print(f"\nFinal answer:\n{final_answer}")
        print('-' * 65)

    return {
        'question'    : question,
        'steps'       : steps,
        'step_results': step_results,
        'final_answer': final_answer,
        'pages'       : sorted(all_pages),
        'latency_ms'  : latency,
    }

# Demo
rag_cot("How do feedback loops in the Arctic amplify global warming?")

["What is Arctic amplification and how does it differ from global average warming rates?", "How does sea ice loss create a positive feedback loop through changes in surface albedo?", "How do rising Arctic temperatures affect permafrost, and what greenhouse gases are released as a result?", "How does increased water vapor and cloud cover in the Arctic contribute to further warming?", "How do these combined Arctic feedback mechanisms accelerate and amplify global warming trends?"]  Step 1/5: What is Arctic amplification and how does it differ from global averag...
The provided context does not cover Arctic amplification or how it differs from global average warming rates. The pages provided focus solely on weather forecasting methods (persistence, trends, climatology, analog, and numerical weather prediction) and do not address polar climate phenomena or comparative warming rates.  Step 2/5: How does sea ice loss create a positive feedback loop through changes ...
The provided context do

{'question': 'How do feedback loops in the Arctic amplify global warming?',
 'steps': ['What is Arctic amplification and how does it differ from global average warming rates?',
  'How does sea ice loss create a positive feedback loop through changes in surface albedo?',
  'How do rising Arctic temperatures affect permafrost, and what greenhouse gases are released as a result?',
  'How does increased water vapor and cloud cover in the Arctic contribute to further warming?',
  'How do these combined Arctic feedback mechanisms accelerate and amplify global warming trends?'],
 'step_results': [{'step': 'What is Arctic amplification and how does it differ from global average warming rates?',
   'retrieval_query': 'What is Arctic amplification and how does it differ from global average warming rates?',
   'hits': [{'text': 'departures of temperature and pressure and other atmospheric conditions from normal\natmospheric conditions for a particular season or period of time. These departures ar

## Step 9 — Comparison: Simple RAG vs CoT RAG

Compare page coverage and answer depth on the same questions.
CoT should cover more pages because later steps retrieve from new parts of the document.

In [9]:
def rag_simple(question: str, top_k: int = TOP_K) -> Dict:
    qvec    = embed_text(question)
    hits    = vs.search(qvec, top_k=top_k)
    context = '\n\n'.join(f"[p.{h['metadata'].get('page_num','?')}]\n{h['text']}" for h in hits)
    answer  = llm(
        f"Answer using ONLY the context. Cite page numbers.\n\n"
        f"Context:\n{context}\n\nQ: {question}\n\nA:"
    )
    return {'hits': hits, 'answer': answer}

test_questions = [
    "What are the mechanisms and consequences of permafrost thaw?",
    "How do ocean currents influence regional climate patterns?",
    "What are the drivers and projected impacts of sea level rise?",
]

print(f"{'Question':<52}  {'Simple pages':>13}  {'CoT pages':>11}  {'Steps':>6}")
print('-' * 90)

for q in test_questions:
    r_simple = rag_simple(q)
    r_cot    = rag_cot(q, verbose=False)

    simple_pages = sorted({h['metadata'].get('page_num','?') for h in r_simple['hits']})
    cot_pages    = r_cot['pages']
    print(f"{q[:51]:<52}  {str(simple_pages):>13}  {str(cot_pages):>11}  {len(r_cot['steps']):>6}")

Question                                               Simple pages    CoT pages   Steps
------------------------------------------------------------------------------------------
The provided context does not contain any information about permafrost thaw, its mechanisms, or its consequences. The context only covers topics related to weather analysis, weather forecasting, and related activities.["What is permafrost and how is it distributed globally?", "What physical and thermal processes cause permafrost to thaw?", "What structural and hydrological changes occur in the landscape as permafrost thaws?", "What biogeochemical processes are triggered by permafrost thaw, particularly regarding carbon and greenhouse gas release?", "What are the broader ecological, infrastructural, and climate feedback consequences of permafrost thaw?"]The provided context does not contain any information about permafrost or its global distribution. The pages given focus exclusively on weather analysis, forec

## Step 10 — Full CoT Demo on a Multi-Layered Question

A question that genuinely requires sequential reasoning — each step's finding
changes what the next step should look for.

In [10]:
rag_cot(
    "What are the primary drivers of global temperature increase, "
    "how do they interact with ocean and ice systems, "
    "and what are the consequences for extreme weather events?",
    top_k=4,
    max_steps=5
)

["What are the primary drivers (forcings) of global temperature increase, such as greenhouse gases, solar variability, and aerosols?", "How do these temperature-driving factors interact with ocean systems, including heat absorption, circulation changes, and sea level rise?", "How do rising temperatures and ocean changes interact with polar ice and glacial systems, including feedback mechanisms?", "What are the combined consequences of ocean warming, ice loss, and altered atmospheric conditions on the frequency and intensity of extreme weather events?"]  Step 1/4: What are the primary drivers (forcings) of global temperature increase...
The provided context does not cover the primary drivers (forcings) of global temperature increase, such as greenhouse gases, solar variability, or aerosols. The excerpts focus solely on weather analysis tools, data collection instruments (radiosondes, satellites, radars), and forecasting processes [p.1], [p.5]. This topic would need to be addressed using

{'question': 'What are the primary drivers of global temperature increase, how do they interact with ocean and ice systems, and what are the consequences for extreme weather events?',
 'steps': ['What are the primary drivers (forcings) of global temperature increase, such as greenhouse gases, solar variability, and aerosols?',
  'How do these temperature-driving factors interact with ocean systems, including heat absorption, circulation changes, and sea level rise?',
  'How do rising temperatures and ocean changes interact with polar ice and glacial systems, including feedback mechanisms?',
  'What are the combined consequences of ocean warming, ice loss, and altered atmospheric conditions on the frequency and intensity of extreme weather events?'],
 'step_results': [{'step': 'What are the primary drivers (forcings) of global temperature increase, such as greenhouse gases, solar variability, and aerosols?',
   'retrieval_query': 'What are the primary drivers (forcings) of global temper

## Step 11 — Summary

| Component | Implementation |
|-----------|---------------|
| Planner | Claude Sonnet 4.6 → JSON array of ordered reasoning steps |
| Retrieval query | Step description + last 2 prior findings (≤300 chars) |
| Per-step reasoning | LLM over retrieved context → 2–3 sentence finding |
| Synthesis | LLM combines all findings into final cohesive answer |
| Vector DB | Qdrant Cloud → OpenSearch → in-memory |
| Embeddings | Amazon Bedrock Titan V2 (1024-dim) |

## How CoT RAG compares to other Tier 3 patterns

| | Query Decomposition (13) | Step-Back (14) | Fusion (15) | **CoT RAG (16)** |
|---|---|---|---|---|
| Retrieval passes | N parallel | 2 parallel | N parallel | N **sequential** |
| Each query informed by prior? | ✗ | ✗ | ✗ | ✓ |
| Best for | Multi-part Qs | Specific Qs needing background | Broad/ambiguous Qs | Multi-layered reasoning Qs |
| Extra LLM calls | 1 decompose | 1 step-back | 1 variant-gen | 1 plan + N reason |
| Latency | Medium | Low | Medium | Highest (sequential) |

## When to use CoT RAG

- Question requires conclusions from step A before you can sensibly search for step B
- Multi-causal chains: mechanism → effect → consequence → implication
- User needs a structured, step-by-step explanation, not just a synthesised answer
- Answering *how* and *why* questions across connected topics

## Limitations

- **Latency**: N sequential LLM + embed calls — cannot be parallelised by design
- **Error propagation**: a wrong finding in step 2 biases all subsequent steps
- **Over-planning**: for simple questions the planner may generate unnecessary steps

### Next: **17 — ReAct RAG**

In [ ]:
# vs.delete_collection()  # uncomment to clean up
print(f"Collection '{COLLECTION_NAME}' in {vs._backend} — {vs.count()} vectors")
print("\nDone. Give the go-ahead for notebook 17.")